# GeoSR-AI — Notebook 04: Advanced Models (EDSR / SwinIR) & Spectral Loss
**Deep Learning Based Super Resolution Mapping from Medium-Resolution Satellite Imagery**

### Overview
This notebook covers:
1. Advanced Deep Learning Super-Resolution Architectures (**EDSR** / **SwinIR**)
2. **Spectral Consistency Loss** ($\mathcal{L}_{spec}$) preserving inter-channel color ratio
3. **Structural Edge Loss** ($\mathcal{L}_{edge}$) preserving roads, building boundaries, and field edges
4. Total Compound Loss formulation:
   $$\text{Total Loss} = \lambda_1 \mathcal{L}_{recon} + \lambda_2 \mathcal{L}_{spec} + \lambda_3 \mathcal{L}_{edge}$$
5. Training framework with mixed precision and residual learning

In [1]:
import os
import sys
import torch
import yaml
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from models.factory import create_model
from losses.combined import GeoSRCombinedLoss
from datasets.paired_dataset import create_dataloaders
from training.trainer import GeoSRTrainer


## 1. Build EDSR / SwinIR Model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
edsr_model = create_model("edsr", in_channels=3, out_channels=3, scale_factor=4, num_blocks=16).to(device)
print(f"EDSR Model Parameters: {sum(p.numel() for p in edsr_model.parameters() if p.requires_grad):,}")

EDSR Model Parameters: 1,517,571


## 2. Test Spectral & Structural Loss

In [3]:
loss_fn = GeoSRCombinedLoss(reconstruction_weight=1.0, spectral_weight=0.1, structural_weight=0.05)
pred_dummy = torch.rand(4, 3, 128, 128, device=device)
target_dummy = torch.rand(4, 3, 128, 128, device=device)

total_loss, loss_dict = loss_fn(pred_dummy, target_dummy)
print(f"Total Compound Loss: {total_loss.item():.4f}")
for k, v in loss_dict.items():
    print(f"  - {k}: {v:.4f}")

Total Compound Loss: 0.4273
  - total_loss: 0.4273
  - reconstruction_loss: 0.3329
  - spectral_loss: 0.5876
  - structural_loss: 0.7127
